# Comparando modelos

## 1 Dependências e configuração

In [ ]:
import sys
sys.path.insert(0, "../src")

from support_module import RANDOM_STATE, generate_credit_risk_base, data_preprocess, profile_for_test_prediction
import mlflow
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score

In [2]:
# gera base de dados sintetica para risco de credito
df = generate_credit_risk_base(6000)
print(df.shape)
df.head()

(6000, 9)


,idade,renda_mensal,tempo_emprego_anos,score_interno,valor_solicitado,num_parcelas,tipo_renda,possui_restricao,inadimplente
0,25,5630.15,0.3,501,5234.99,24,CLT,nao,0
1,56,2491.57,9.6,593,8264.73,18,CLT,sim,0
2,50,1020.17,1.0,695,12435.70,12,CLT,nao,0
3,41,1580.32,11.7,527,14883.12,48,CLT,nao,0
4,41,6635.91,2.7,618,12856.88,24,CLT,nao,0


In [3]:
# seleciona features
TARGET   = "inadimplente"
FEATURES = [c for c in df.columns if c != TARGET]
x        = df[FEATURES]
y        = df[TARGET]

# separa colunas numericas e categoricas
num_cols = df[FEATURES].select_dtypes(include='number').columns
cat_cols = [f for f in FEATURES if f not in num_cols]

# separa dados de treino e teste, seguindo proporcao de casos positivos e negativos do target
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)

# pipeline de transformacao dos daddos (disponivel em `src/support_module.py`)
preprocess = data_preprocess()

x_train.shape, x_test.shape

((4500, 8), (1500, 8))

## 2 Registrando diferentes modelos com autolog

O autolog é um recurso do MLFlow que automatiza o processo _logging_ durante o treinamento de modelos. Em vez de definirmos manualmente cada comando para salvar um parâmetro ou métrica, o MLFlow "observa" o treinamento e realiza esse registro de forma automática.

O MLFlow define o que registrar de acordo com a biblioteca sendo utilizada. Usando o `scikit-learn`, por exemplo, o MLFlow utiliza métodos internos como o `get_params` para identificar automaticamente quais são os parâmetros do modelo e componentes do pipeline (como o pré-processador). A classe do classificador também é identificada e, por padrão, o modelo ajustado é armazenado como um artefato.

É importante mencionar que, apesar de o autolog ser muito bom para registro técnico automático, o registro manual continua sendo insubstituível para métricas de negócio específicas ou em situações em que é necessário ter controle total sobre o que é exibido na tabela de comparação. Por isso, é muito comum adotar uma abordagem híbrida.

Além disso, como são registradas muitas informações automaticamente, pode acontecer de ele acabar "poluindo" o experimento com excesso de _logs_. Por isso, pode fazer sentido desativar o autolog em momentos específicos e ativá-lo apenas no momento do treinamento principal que se deseja rastrear.

Para ilustrar os benefícios do autolog, vamos criar um novo experimento e comparar três modelos diferentes: o _baseline_, de Regressão Logística, e mais dois _challengers_, um do tipo Árvore de Decisão e outro Random Forest. Dessa forma, temos um ponto de partida conhecido (_baseline_) e podemos avaliar se os outros modelos (de tipos diferentes) trazem algum ganho real de performance. Usando o autolog, serão capturadas informações técnicas de cada modelo de modo que será possível realizar uma comparação justa entre eles.

In [4]:
# define o tracking com o banco de dados relacional ja existente
mlflow.set_tracking_uri("sqlite:///../mlflow.db")


# cria experimento e define local para armazenar artefatos
EXPERIMENT_NAME = "risco_credito_comparacao_modelos"
mlflow.create_experiment(name=EXPERIMENT_NAME, artifact_location="/home/bruno/estudos/data-science-portifolio/mlflow/mlruns/1/")

# define experimento criado como ativo
mlflow.set_experiment(EXPERIMENT_NAME)

<Experiment: artifact_location='/home/bruno/estudos/data-science-portifolio/mlflow/mlruns/1/', creation_time=1791835030069, effective_trace_archival_retention=None, experiment_id='3', last_update_time=1791835030069, lifecycle_stage='active', name='risco_credito_comparacao_modelos', tags={}, trace_location=None, workspace='default'>

Abaixo definimos os modelos candidatos e os executamos com autolog:

In [5]:
# define modelos candidatos e seus hiperparametros
model_candidates = {
    "logistic_regression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
    "decision_tree": DecisionTreeClassifier(
        max_depth=5,
        min_samples_leaf=50,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=8,
        min_samples_leaf=20,
        class_weight="balanced",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
}

In [6]:
# rodando validacao cruzada e treino final de cada modelo candidato
results  = []
CV_FOLDS = 3

for model_name, estimator in model_candidates.items():
    print(f"Treinando: {model_name}")

    # validacao cruzada
    mlflow.sklearn.autolog(disable=True) # desabilita autolog para evitar poluir logs
    cv_pipe = Pipeline(
        steps=[
            ("prep", data_preprocess()),
            ("model", clone(estimator)),
        ]
    )
    cv        = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    cv_scores = cross_val_score(cv_pipe, x_train, y_train, cv=cv, scoring="roc_auc", n_jobs=1)


    # ativa autolog apenas para o treino final da run
    mlflow.sklearn.autolog(
        log_input_examples=False,
        log_model_signatures=True,
        log_models=True,
        silent=True,
    )

    run_name = f"hibrid_log_comparison_{model_name}"
    
    with mlflow.start_run(run_name=run_name) as run:
        model_pipe = Pipeline(
            steps=[
                ("prep", data_preprocess()),
                ("model", clone(estimator)),
            ]
        )

        # treina o modelo e faz previsoes no conjunto de teste
        model_pipe.fit(x_train, y_train)
        y_proba = model_pipe.predict_proba(x_test)[:, 1]
        y_pred  = model_pipe.predict(x_test)

        # registra parametros utilizados no classificador
        mlflow.log_param("model_name", model_name)
        mlflow.log_param("cv_folds", CV_FOLDS)
        mlflow.log_param("dataset_version", "synthetic_credit_v1")
        mlflow.log_param("target", TARGET)

        # registra metricas do modelo
        metrics = {
            "roc_auc_cv_media":       float(cv_scores.mean()),
            "roc_auc_cv_desvio":      float(cv_scores.std()),
            "roc_auc_teste":          float(roc_auc_score(y_test, y_proba)),
            "accuracy_teste":         float(accuracy_score(y_test, y_pred)),
            "precision_inadimplente": float(precision_score(y_test, y_pred, zero_division=0)),
            "recall_inadimplente":    float(recall_score(y_test, y_pred, zero_division=0)),
            "f1_inadimplente":        float(f1_score(y_test, y_pred, zero_division=0)),
        }
        mlflow.log_metrics(metrics)

        # adiciona tags para melhor rastreabilidade da run
        mlflow.set_tags({
            "project":       "risco_credito",
            "tracking_type": "autolog_plus_manual_metrics",
            "task":          "binary_classification",
            "model_name":    model_name,
            "run_purpose":   "compare_model_candidates",
        })

        mlflow.log_dict(
            {
                "model_name": model_name,
                "run_id":     run.info.run_id,
                "metrics":    metrics,
                "notes":      "Run gerada com mlflow.sklearn.autolog e métricas de teste registradas manualmente para comparação.",
            }, 
            "reports/evaluation_summary.json"
        )

        results.append(
            {
                "run_id": run.info.run_id,
                "run_name": run_name,
                "model_name": model_name,
                **metrics,
            }
        )

    # desativa autolog depois da run, para evitar logging automatico acidental
    mlflow.sklearn.autolog(disable=True)

comparison_df = (
    pd.DataFrame(results)
    .sort_values("roc_auc_teste", ascending=False)
    .reset_index(drop=True)
)

comparison_df

Treinando: logistic_regression


/home/bruno/estudos/data-science-portifolio/mlflow/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:3424: FutureWarning: `y_pred` was renamed to `y_proba` in version 1.9 and will be removed in 1.11. Use `y_proba` instead.
  warnings.warn(


Treinando: decision_tree


/home/bruno/estudos/data-science-portifolio/mlflow/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:3424: FutureWarning: `y_pred` was renamed to `y_proba` in version 1.9 and will be removed in 1.11. Use `y_proba` instead.
  warnings.warn(


Treinando: random_forest


/home/bruno/estudos/data-science-portifolio/mlflow/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:3424: FutureWarning: `y_pred` was renamed to `y_proba` in version 1.9 and will be removed in 1.11. Use `y_proba` instead.
  warnings.warn(


,run_id,run_name,model_name,roc_auc_cv_media,roc_auc_cv_desvio,roc_auc_teste,accuracy_teste,precision_inadimplente,recall_inadimplente,f1_inadimplente
0,e4afeca8be834b3fb69b1dbd6e3f4c0f,hibrid_log_comparison_logistic_regression,logistic_regression,0.902266,0.001095,0.903577,0.822667,0.389034,0.823204,0.528369
1,9ca31560460247d38490485103e1a03f,hibrid_log_comparison_random_forest,random_forest,0.889951,0.001628,0.894131,0.803333,0.364929,0.850829,0.510779
2,7a7bb50e45434c29a3840190f02460f4,hibrid_log_comparison_decision_tree,decision_tree,0.875530,0.004451,0.890569,0.800667,0.362791,0.861878,0.510638


Note que acima utilizamos um abordagem híbrida para registro dos _logs_, mantendo o autolog ativado durante o treinamento do modelo e definindo manualmente o registro de outras métricas. E a cada iterção, salvamos um contentor com as métricas definidas manualmente, que ao final foram utilizados para gerar um DataFrame de resumo para comparação dos modelos.

Outro ponto interessante de mencionar é que os modelos treinados foram salvos em `mlruns/1/models/`, e podemos usar a API do MLFlow para carregar esse modelos e realizar predições:

In [7]:
novo = profile_for_test_prediction()

for i, row in comparison_df.iterrows():
    model_uri = f"runs:/{row['run_id']}/model"
    model_i   = mlflow.sklearn.load_model(model_uri)

    print(f"- Modelo {row['model_name']}")
    print(f"\tProbabilidade estimada de inadimplencia: {model_i.predict_proba(novo)[0, 1]:.1%}")
    print("\tRotulo binario (limiar 0.5):", int(model_i.predict(novo)[0]))

- Modelo logistic_regression
	Probabilidade estimada de inadimplencia: 27.2%
	Rotulo binario (limiar 0.5): 0
- Modelo random_forest
	Probabilidade estimada de inadimplencia: 27.2%
	Rotulo binario (limiar 0.5): 0
- Modelo decision_tree
	Probabilidade estimada de inadimplencia: 21.3%
	Rotulo binario (limiar 0.5): 0


Ao carregar o modelo usando a URI `runs:/`, o MLflow carrega o objeto do modelo garantindo que todas as dependências e configurações do modelo estejam configuradas conforme foram registradas durante o treinamento, evitando problemas de compatibilidade de versões de bibliotecas.

## 3 Comparando _runs_

A partir de UI do MLFlow podemos selecionar execuções específicas e compará-las lado a lado com auxílio de recursos visuais, podendo escolher exatamente as métricas que queremos avaliar. Isso também pode ser feito com código, usando a API do MLFlow, que disponibiliza o método `search_runs()`.

O importante é que esse processo de tomada de decisão seja feita através de escolhas feitas em acordo com critérios objetivos.

Nesse projeto, a métrica principal é **ROC AUC** (Área Sob a Curva ROC), que é uma medida da capacidade do modelo de discriminar entre as classes (quem é inadimplente e quem não é), é como se fosse uma "nota geral" para o modelo. Considerando que o modelo atribui uma probabilidade de risco para cada cliente, ao escolhermos um ponto de corte (_threshold_) para decidir quem reprovar, o ROC AUC permite entender quão bem o modelo consegue separar esses dois grupos para diferentes pontos de corte possíveis. Quanto mais perto de $1,0$, melhor o modelo está em ordenar os clientes, colocando os de maior risco no topo da lista. Se estivéssemos apenas chutando, teríamos um valor próximo de $0,5$.

Mas também precisamos nos atentar a outras métricas "secundárias" (nesse projeto), que são:

- **Recall**: medida da sensibilidade do modelo para capturar casos positivos.
  - Pode ser entendida como: _"de todos casos positivos, quantos o modelo acertou"_, ou seja, é a fração de positivos identificados pelo modelo.
  - Sua fórmula é: $\text{recall} = \frac{\text{TP}}{\text{TP}+\text{FN}}$. Ou seja, quanto mais perto de $1$, mais casos positivos (clientes inadimplentes) foram indetificados.
  - Em problemas de risco de crédito é uma métrica crucial, porque se não identificarmos um cliente inadimplente (falso negativo), o prejuízo financeiro é alto. Queremos um modelo que capture o máximo possível de casos positivos.
- **Precision**: determina em que medida as predições positivas são precisas.
  - Pode ser entendida como: _"de todos classificados como positivos pelo modelo, quantos ele acertou?"_. 
  - Sua fórmula é: $\text{precision} = \frac{\text{TP}}{\text{TP}+\text{FP}}$. Ou seja, quanto mais próximo de $1$, maior o nível de acerto do modelo dentre os clientes classificados como positivos (inadimplentes), e quanto mais próximo de $0$, menos clientes classificados como inadimplentes eram de fato inadimplentes.
  - É o controle para não sermos rigorosos demais: se tivermos muitos falsos positivos, podemos acabar recusando clientes que pagariam suas contas corretamente, o que gera perda de oportunidade de negócio ou custo operacional de análise manual desnecessária.

A escolha do modelo ideal não se trata apenas de selecionar o modelo que apresenta o maior valor para ROC, mas sobre equilibrar essas métricas para atender ao objetivo de negócio.

In [12]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

df_runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="tags.run_purpose = 'compare_model_candidates'",
    order_by=["metrics.roc_auc_teste DESC"],
)

df_runs.shape

(3, 115)

In [17]:
df_runs.columns

Index(['run_id', 'experiment_id', 'status', 'artifact_uri', 'start_time',
       'end_time', 'metrics.training_score', 'metrics.recall_inadimplente',
       'metrics.precision_inadimplente', 'metrics.training_log_loss',
       ...
       'tags.mlflow.runName', 'tags.course_lesson', 'tags.estimator_name',
       'tags.task', 'tags.run_purpose', 'tags.project', 'tags.tracking_type',
       'tags.estimator_class', 'tags.mlflow.source.type',
       'tags.mlflow.source.name'],
      dtype='str', length=115)

O DataFrame retornado possui mais de 100 métricas, por isso devemos selecionar apenas as métricas de maior interesse, de modo a facilitar o processo de decisão:

In [13]:
# monta tabela de comparacao selecionando apenas colunas de interesse, e ordenando pelo desempenho no conjunto de teste
comparison_cols = [
    "run_id",
    "tags.mlflow.runName",
    "params.model_name",
    "metrics.roc_auc_teste",
    "metrics.roc_auc_cv_media",
    "metrics.roc_auc_cv_desvio",
    "metrics.accuracy_teste",
    "metrics.precision_inadimplente",
    "metrics.recall_inadimplente",
    "metrics.f1_inadimplente",
]

available_cols = [col for col in comparison_cols if col in df_runs.columns]

df_ranking = (
    df_runs[available_cols]
    .sort_values("metrics.roc_auc_teste", ascending=False)
    .reset_index(drop=True)
)

df_ranking

,run_id,tags.mlflow.runName,params.model_name,metrics.roc_auc_teste,metrics.roc_auc_cv_media,metrics.roc_auc_cv_desvio,metrics.accuracy_teste,metrics.precision_inadimplente,metrics.recall_inadimplente,metrics.f1_inadimplente
0,e4afeca8be834b3fb69b1dbd6e3f4c0f,hibrid_log_comparison_logistic_regression,logistic_regression,0.903577,0.902266,0.001095,0.822667,0.389034,0.823204,0.528369
1,9ca31560460247d38490485103e1a03f,hibrid_log_comparison_random_forest,random_forest,0.894131,0.889951,0.001628,0.803333,0.364929,0.850829,0.510779
2,7a7bb50e45434c29a3840190f02460f4,hibrid_log_comparison_decision_tree,decision_tree,0.890569,0.875530,0.004451,0.800667,0.362791,0.861878,0.510638


In [14]:
# aplica criterios para selecionar apenas os modelos candidatos com desempenho satisfatorio
df_shortlist = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string=(
        "tags.run_purpose = 'compare_model_candidates' "
        "and metrics.roc_auc_teste >= 0.85 "
        "and metrics.recall_inadimplente >= 0.85"
    ),
    order_by=["metrics.roc_auc_teste DESC"],
)

shortlist_cols = [col for col in comparison_cols if col in df_shortlist.columns]

df_shortlist[shortlist_cols]

,run_id,tags.mlflow.runName,params.model_name,metrics.roc_auc_teste,metrics.roc_auc_cv_media,metrics.roc_auc_cv_desvio,metrics.accuracy_teste,metrics.precision_inadimplente,metrics.recall_inadimplente,metrics.f1_inadimplente
0,9ca31560460247d38490485103e1a03f,hibrid_log_comparison_random_forest,random_forest,0.894131,0.889951,0.001628,0.803333,0.364929,0.850829,0.510779
1,7a7bb50e45434c29a3840190f02460f4,hibrid_log_comparison_decision_tree,decision_tree,0.890569,0.875530,0.004451,0.800667,0.362791,0.861878,0.510638


In [ ]:
# cria label para identificar modelo mais bem ranqueado: candidato a campeao
ranking_with_roles                   = df_ranking.copy()
ranking_with_roles["candidate_role"] = ["champion_candidate" if i == 0 else "challenger" for i in range(len(ranking_with_roles))]
ranking_with_roles

,run_id,tags.mlflow.runName,params.model_name,metrics.roc_auc_teste,metrics.roc_auc_cv_media,metrics.roc_auc_cv_desvio,metrics.accuracy_teste,metrics.precision_inadimplente,metrics.recall_inadimplente,metrics.f1_inadimplente,candidate_role
0,e4afeca8be834b3fb69b1dbd6e3f4c0f,hibrid_log_comparison_logistic_regression,logistic_regression,0.903577,0.902266,0.001095,0.822667,0.389034,0.823204,0.528369,champion_candidate
1,9ca31560460247d38490485103e1a03f,hibrid_log_comparison_random_forest,random_forest,0.894131,0.889951,0.001628,0.803333,0.364929,0.850829,0.510779,challenger
2,7a7bb50e45434c29a3840190f02460f4,hibrid_log_comparison_decision_tree,decision_tree,0.890569,0.875530,0.004451,0.800667,0.362791,0.861878,0.510638,challenger


In [16]:
# exporta ranking para CSV, pode ser adicionado como artefato do experimento
df_ranking.to_csv("3_ranking_comparacao_runs.csv", index=False)